In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("THANH CONG - Python da san sang!")
print("Pandas:", pd.__version__)
print("Thu muc:", Path.cwd())


THANH CONG - Python da san sang!
Pandas: 3.0.6
Thu muc: c:\Users\phat\Downloads\covid19-time-series-ml\notebooks


In [2]:

# PHAN 1 - DOC VA KIEM TRA DU LIEU COVID-19

# Duong dan den thu muc data
data_dir = Path.cwd().parent / "data"

# Danh sach 3 file CSV
files = {
    "Bac My": "covid19_north_america.csv",
    "Chau A": "covid19_asia.csv",
    "Chau Phi": "covid19_africa.csv"
}

# Luu du lieu cua tung chau luc
datasets = {}

for region, filename in files.items():
    path = data_dir / filename

    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(
        df["Date"], errors="coerce"
    )

    datasets[region] = df

    print("=" * 50)
    print("KHU VUC:", region)
    print("So dong:", df.shape[0])
    print("So cot:", df.shape[1])
    print("So quoc gia:",
          df["Country/Region"].nunique())
    print("Ngay bat dau:", df["Date"].min())
    print("Ngay ket thuc:", df["Date"].max())
    print("So dong thieu ngay:",
          df["Date"].isna().sum())

    print("\n5 dong dau tien:")
    print(df.head())

print("\nDA DOC XONG 3 BO DU LIEU!")


KHU VUC: Bac My
So dong: 6580
So cot: 10
So quoc gia: 24
Ngay bat dau: 2020-01-22 00:00:00
Ngay ket thuc: 2020-07-27 00:00:00
So dong thieu ngay: 0

5 dong dau tien:
     Province/State       Country/Region        Lat        Long       Date  \
0               NaN  Antigua and Barbuda  17.060800  -61.796400 2020-01-22   
1               NaN              Bahamas  25.025885  -78.035889 2020-01-22   
2               NaN             Barbados  13.193900  -59.543200 2020-01-22   
3           Alberta               Canada  53.933300 -116.576500 2020-01-22   
4  British Columbia               Canada  53.726700 -127.647600 2020-01-22   

   Confirmed  Deaths  Recovered  Active WHO Region  
0          0       0          0       0   Americas  
1          0       0          0       0   Americas  
2          0       0          0       0   Americas  
3          0       0          0       0   Americas  
4          0       0          0       0   Americas  
KHU VUC: Chau A
So dong: 15040
So cot: 10
So qu

In [3]:

# PHAN 2 - KIEM TRA CHAT LUONG DU LIEU

results = []

for region, df in datasets.items():

    important_cols = [
        "Date", "Country/Region",
        "Confirmed", "Deaths",
        "Recovered", "Active"
    ]

    results.append({
        "Khu vuc": region,
        "So dong": len(df),
        "So quoc gia": df["Country/Region"].nunique(),
        "So ngay": df["Date"].nunique(),
        "Dong trung hoan toan": df.duplicated().sum(),
        "Gia tri thieu quan trong":
            df[important_cols].isna().sum().sum(),
        "So ca nhiem am": (df["Confirmed"] < 0).sum(),
        "So ca tu vong am": (df["Deaths"] < 0).sum()
    })

quality_report = pd.DataFrame(results)

display(quality_report)


,Khu vuc,So dong,So quoc gia,So ngay,Dong trung hoan toan,Gia tri thieu quan trong,So ca nhiem am,So ca tu vong am
0,Bac My,6580,24,188,0,0,0,0
1,Chau A,15040,48,188,0,0,0,0
2,Chau Phi,10340,55,188,0,0,0,0


In [4]:

# PHAN 3 - KIEM TRA DU LIEU THEO NGAY

check_results = []

for region, df in datasets.items():

    # Tong hop so ca theo tung ngay
    daily = (
        df.groupby("Date")[
            ["Confirmed", "Deaths",
             "Recovered", "Active"]
        ]
        .sum()
        .sort_index()
    )

    # Tinh so ca phat sinh moi ngay
    daily["New_Confirmed"] = (
        daily["Confirmed"].diff()
    )
    daily["New_Deaths"] = (
        daily["Deaths"].diff()
    )
    daily["New_Recovered"] = (
        daily["Recovered"].diff()
    )

    # Kiem tra ngay bi thieu
    expected_dates = pd.date_range(
        daily.index.min(),
        daily.index.max(),
        freq="D"
    )

    missing_dates = expected_dates.difference(
        daily.index
    )

    # Luu ket qua kiem tra
    check_results.append({
        "Khu vuc": region,
        "Ngay thieu": len(missing_dates),
        "New_Confirmed am":
            (daily["New_Confirmed"] < 0).sum(),
        "New_Deaths am":
            (daily["New_Deaths"] < 0).sum(),
        "New_Recovered am":
            (daily["New_Recovered"] < 0).sum(),
        "Sai lech Active": (
            daily["Active"] != (
                daily["Confirmed"]
                - daily["Deaths"]
                - daily["Recovered"]
            )
        ).sum()
    })

daily_quality = pd.DataFrame(check_results)

display(daily_quality)


,Khu vuc,Ngay thieu,New_Confirmed am,New_Deaths am,New_Recovered am,Sai lech Active
0,Bac My,0,0,0,1,0
1,Chau A,0,0,0,0,0
2,Chau Phi,0,0,0,3,0


In [5]:

# PHAN 4 - TIM NGAY CO SO CA HOI PHUC AM

abnormal_rows = []

for region, df in datasets.items():

    daily = (
        df.groupby("Date")["Recovered"]
        .sum()
        .sort_index()
    )

    changes = daily.diff()

    for date in changes[changes < 0].index:
        previous_date = date - pd.Timedelta(days=1)

        abnormal_rows.append({
            "Khu vuc": region,
            "Ngay": date.date(),
            "Recovered ngay truoc": daily.get(previous_date),
            "Recovered hien tai": daily.loc[date],
            "Thay doi": changes.loc[date]
        })

abnormal_report = pd.DataFrame(abnormal_rows)

display(abnormal_report)


,Khu vuc,Ngay,Recovered ngay truoc,Recovered hien tai,Thay doi
0,Bac My,2020-07-19,1492192,1486137,-6055.0
1,Chau Phi,2020-03-15,41,35,-6.0
2,Chau Phi,2020-03-24,143,137,-6.0
3,Chau Phi,2020-03-26,211,184,-27.0


In [6]:

# PHAN 5 - DOI CHIEU DU LIEU TONG HOP

region_names = {
    "Bac My": "North America",
    "Chau A": "Asia",
    "Chau Phi": "Africa"
}

columns = [
    "Confirmed", "Deaths",
    "Recovered", "Active"
]

# Tong hop lai tu 3 file goc
parts = []

for region, df in datasets.items():
    daily = df.groupby("Date")[columns].sum()
    daily = daily.reset_index()
    daily["Continent"] = region_names[region]
    parts.append(daily)

calculated = pd.concat(parts, ignore_index=True)

# Doc file tong hop cua nhom
existing = pd.read_csv(
    data_dir / "covid19_continent_daily_totals.csv"
)
existing["Date"] = pd.to_datetime(existing["Date"])

# Doi chieu theo ngay va chau luc
comparison = calculated.merge(
    existing,
    on=["Date", "Continent"],
    how="outer",
    suffixes=("_new", "_old"),
    indicator=True
)

print("So dong doi chieu:", len(comparison))
print("Trang thai ghep:")
print(comparison["_merge"].value_counts())

for col in columns:
    a = comparison[col + "_new"]
    b = comparison[col + "_old"]

    mismatch = ~np.isclose(
        a, b, equal_nan=True
    )

    print(col, "- So dong khac:", mismatch.sum())


So dong doi chieu: 564
Trang thai ghep:
_merge
both          564
left_only       0
right_only      0
Name: count, dtype: int64
Confirmed - So dong khac: 0
Deaths - So dong khac: 0
Recovered - So dong khac: 0
Active - So dong khac: 0


In [8]:

# PHAN 6 - KIEM TRA CAC COT DA TINH TOAN

checks = []

for continent, group in existing.groupby("Continent"):
    group = group.sort_values("Date").copy()

    # So ca nhiem moi duoc tinh lai
    new_cases = group["Confirmed"].diff().fillna(0)

    # Trung binh truot tu so ca moi
    rolling_7 = new_cases.rolling(
        window=7, min_periods=1
    ).mean()

    rolling_14 = new_cases.rolling(
        window=14, min_periods=1
    ).mean()

    checks.append({
        "Continent": continent,
        "New_Confirmed khac": (
            ~np.isclose(
                group["New_Confirmed"],
                new_cases
            )
        ).sum(),
        "Rolling_7D khac": (
            ~np.isclose(
                group["Rolling_7D_New_Confirmed"],
                rolling_7
            )
        ).sum(),
        "Rolling_14D khac": (
            ~np.isclose(
                group["Rolling_14D_New_Confirmed"],
                rolling_14
            )
        ).sum(),
        "Cumulative khac": (
            ~np.isclose(
                group["Cumulative_Confirmed_Total"],
                group["Confirmed"]
            )
        ).sum()
    })

verification = pd.DataFrame(checks)
display(verification)


,Continent,New_Confirmed khac,Rolling_7D khac,Rolling_14D khac,Cumulative khac
0,Africa,0,0,0,0
1,Asia,0,0,0,0
2,North America,0,0,0,0


In [9]:

# PHAN 7 - KIEM TRA PHAN LOAI CHAU LUC

from itertools import combinations

continent_names = {
    "Bac My": "North America",
    "Chau A": "Asia",
    "Chau Phi": "Africa"
}

country_sets = {}
audit_parts = []

for region, df in datasets.items():
    countries = set(
        df["Country/Region"].dropna().unique()
    )

    country_sets[region] = countries

    print(region, ":", len(countries), "quoc gia")

    part = df[
        ["Country/Region", "WHO Region"]
    ].drop_duplicates().copy()

    part["Continent"] = continent_names[region]
    audit_parts.append(part)

# Kiem tra quoc gia nam trong nhieu bo
for a, b in combinations(country_sets, 2):
    overlap = country_sets[a] & country_sets[b]
    print(a, "va", b, "- Trung:", sorted(overlap))

# Danh sach can kiem tra phan vung
audit = pd.concat(audit_parts, ignore_index=True)

special_countries = [
    "Armenia", "Azerbaijan", "Georgia",
    "Turkey", "Russia", "Kazakhstan",
    "Cyprus", "Egypt"
]

print("\nQUOC GIA CAN KIEM TRA:")
display(
    audit[
        audit["Country/Region"].isin(special_countries)
    ].sort_values("Country/Region")
)


Bac My : 24 quoc gia
Chau A : 48 quoc gia
Chau Phi : 55 quoc gia
Bac My va Chau A - Trung: []
Bac My va Chau Phi - Trung: []
Chau A va Chau Phi - Trung: []

QUOC GIA CAN KIEM TRA:


,Country/Region,WHO Region,Continent
25,Armenia,Europe,Asia
26,Azerbaijan,Europe,Asia
33,Cyprus,Europe,Asia
84,Egypt,Eastern Mediterranean,Africa
34,Georgia,Europe,Asia
42,Kazakhstan,Europe,Asia
55,Russia,Europe,Asia
61,Turkey,Europe,Asia


In [10]:

# PHAN 8 - KIEM TRA SO NGAY CUA TUNG QUOC GIA

missing_country_days = []

for region, df in datasets.items():

    total_days = df["Date"].nunique()

    country_days = (
        df.groupby("Country/Region")["Date"]
        .nunique()
    )

    for country, days in country_days.items():

        if days < total_days:
            missing_country_days.append({
                "Khu vuc": region,
                "Quoc gia": country,
                "So ngay co du lieu": days,
                "So ngay thieu": total_days - days
            })

coverage_report = pd.DataFrame(missing_country_days)

if coverage_report.empty:
    print("THANH CONG!")
    print("Tat ca quoc gia co du 188 ngay.")
else:
    print("Co quoc gia thieu ngay:")
    display(coverage_report)


THANH CONG!
Tat ca quoc gia co du 188 ngay.



## KẾT QUẢ PHẦN 1 — DATA PREPARATION

### 1. Dữ liệu nghiên cứu

Nghiên cứu sử dụng dữ liệu COVID-19 của 3 khu vực:
- Bắc Mỹ: 6.580 bản ghi, 24 quốc gia/vùng lãnh thổ.
- Châu Á: 15.040 bản ghi, 48 quốc gia/vùng lãnh thổ.
- Châu Phi: 10.340 bản ghi, 55 quốc gia/vùng lãnh thổ.

Thời gian: 22/01/2020 đến 27/07/2020.

### 2. Kiểm tra chất lượng

- Không phát hiện dòng trùng hoàn toàn.
- Không thiếu dữ liệu trong các cột quan trọng đã kiểm tra.
- Mỗi quốc gia có bản ghi trong đủ 188 ngày.
- Có 4 ngày số ca hồi phục tích lũy giảm.
- Dữ liệu tổng hợp gồm 564 dòng, khớp với dữ liệu nguồn.

### 3. Kiểm tra các biến tính toán

Các cột New_Confirmed, trung bình trượt 7 ngày,
trung bình trượt 14 ngày và số ca tích lũy
đều khớp với kết quả tính lại.

### 4. Lưu ý

Một số quốc gia như Russia, Turkey, Kazakhstan
có lãnh thổ thuộc nhiều châu lục.

Cần thống nhất quy tắc phân loại quốc gia
trước khi diễn giải kết quả nghiên cứu.

Các số ca hồi phục giảm được giữ nguyên
và ghi nhận là bất thường cần kiểm tra.

### 5. Kết luận

Dữ liệu đã vượt qua các bước kiểm tra tính đầy đủ,
tính nhất quán và đối chiếu tổng hợp ban đầu.

Có thể chuyển dữ liệu cho bước phân tích EDA
sau khi nhóm xác nhận quy tắc phân chia châu lục.
